# Sakura Bloom Dynamics — Temporal & Phase-Space Plots (Kaggle/JMA, 102 stations)

Generates, for every station in the Kaggle/JMA sakura dataset (1953–2025):

1. **Temporal plots**: `x1` vs `t` (first bloom day-of-year vs year), `x2` vs `t` (full bloom day-of-year vs year)
2. **Phase-space plots**: `dx1/dt` vs `x1`, `dx2/dt` vs `x2`

organized into PNG files, plus a manifest CSV summarizing data coverage and trends per station.

> Scope note: this notebook covers the **Kaggle/JMA 102-station dataset only**. Kyoto's separate long-term
> Aono historical record (~812 AD onward) is a different source and is intentionally **not** included here.

## 1. Configuration — set your file paths here

In [ ]:
# ==================== CONFIGURATION ====================
# Edit these three paths/settings before running.

FIRST_BLOOM_CSV = "sakura_first_bloom_dates.csv"   # path to the first-bloom-dates CSV
FULL_BLOOM_CSV  = "sakura_full_bloom_dates.csv"    # path to the full-bloom-dates CSV
OUTPUT_DIR      = "sakura_plots"                    # output folder for the organized PNGs + manifest
# =========================================================


## 2. Imports

In [ ]:
import os
import re
from datetime import date, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

plt.rcParams.update({
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 130,
})


## 3. Data utilities

- `normalized_doy`: maps every calendar date onto a fixed non-leap reference year (2001) before
  computing day-of-year, so a given calendar date always gets the same numeric x-value regardless
  of whether that particular year was a leap year. This avoids injecting a spurious ±1 day artifact
  into the estimated derivative `dx/dt`.
- `compute_dxdt`: uses `np.gradient(x, t)` with the *actual* year values, correctly handling stations
  with missing years (gaps) rather than assuming uniform 1-year spacing.

In [ ]:
def get_year_columns(df):
    """Return the list of column names that are purely numeric (i.e. year columns)."""
    return [c for c in df.columns if c.strip().isdigit()]


def normalized_doy(dt_obj):
    """Leap-year-normalized day-of-year (Feb 29 -> 59.5)."""
    if pd.isna(dt_obj):
        return np.nan
    m, d = dt_obj.month, dt_obj.day
    if m == 2 and d == 29:
        return 59.5
    return date(2001, m, d).timetuple().tm_yday  # 2001: fixed non-leap reference year


def extract_station_series(df, station_name, year_cols):
    """Return (years, doy) sorted numpy arrays for one station, with missing years dropped."""
    match = df.loc[df['Site Name'] == station_name]
    if match.empty:
        return np.array([]), np.array([])
    row = match.iloc[0]

    years, doys = [], []
    for yc in year_cols:
        raw = row[yc]
        if pd.isna(raw) or raw == '':
            continue
        dt_obj = pd.to_datetime(raw, errors='coerce')
        if pd.isna(dt_obj):
            continue
        years.append(int(yc))
        doys.append(normalized_doy(dt_obj))

    years = np.array(years, dtype=int)
    doys = np.array(doys, dtype=float)
    order = np.argsort(years)
    return years[order], doys[order]


def parse_30yr_avg(avg_str):
    """Parse a '30 Year Average 1991-2020' cell like '5 13' (month day) into a normalized DOY."""
    if pd.isna(avg_str):
        return None
    s = str(avg_str).strip()
    if s in ('-', ''):
        return None
    parts = s.split()
    if len(parts) != 2:
        return None
    try:
        m, d = int(parts[0]), int(parts[1])
        return date(2001, m, d).timetuple().tm_yday
    except ValueError:
        return None


def species_note(df, station_name):
    """Return the free-text 'Notes' field for a station (species/observation caveats), or None."""
    match = df.loc[df['Site Name'] == station_name]
    if match.empty:
        return None
    val = match.iloc[0].get('Notes', None)
    if pd.isna(val):
        return None
    return str(val)


def linear_trend(years, values):
    """Least-squares linear fit. Returns (slope_per_decade, intercept, fit_years, fit_values) or None."""
    if len(years) < 2:
        return None
    slope, intercept = np.polyfit(years, values, 1)
    fit_years = np.array([years.min(), years.max()], dtype=float)
    fit_values = slope * fit_years + intercept
    return slope * 10.0, intercept, fit_years, fit_values


def compute_dxdt(years, values):
    """Numerical derivative dx/dt via np.gradient (handles non-uniform year spacing)."""
    if len(years) < 2:
        return np.array([])
    return np.gradient(values, years.astype(float))


## 4. Plotting functions

In [ ]:
def safe_stub(name):
    """Turn a station name into a filesystem-safe filename stub."""
    return re.sub(r'[^A-Za-z0-9_-]+', '_', name).strip('_')


def doy_formatter():
    """Matplotlib tick formatter converting normalized-DOY floats back to 'Mon DD' labels."""
    base = date(2001, 1, 1)

    def _fmt(y, _pos):
        try:
            d = base + timedelta(days=float(y) - 1)
            return d.strftime('%b %d')
        except (OverflowError, ValueError):
            return ''
    return FuncFormatter(_fmt)


def make_temporal_plot(station, idx, years1, doy1, years2, doy2, avg1, avg2, note, outdir):
    fig, axes = plt.subplots(2, 1, figsize=(8.5, 7.5))

    panels = [
        (axes[0], years1, doy1, avg1, 'First Bloom (x1)', 'tab:pink'),
        (axes[1], years2, doy2, avg2, 'Full Bloom (x2)', 'tab:red'),
    ]
    for ax, years, doy, avg, label, color in panels:
        if len(years) == 0:
            ax.text(0.5, 0.5, 'No data available', ha='center', va='center',
                     transform=ax.transAxes, fontsize=10, color='gray')
            ax.set_title(f'{label} vs Year')
            continue

        ax.plot(years, doy, 'o-', color=color, ms=4, lw=1, label=label)

        trend = linear_trend(years, doy)
        if trend is not None:
            slope_decade, _intercept, fy, fv = trend
            direction = 'earlier' if slope_decade < 0 else 'later'
            ax.plot(fy, fv, '--', color='k', lw=1.2, alpha=0.7,
                     label=f'Trend: {slope_decade:+.2f} days/decade ({direction})')

        if avg is not None:
            ax.axhline(avg, color='gray', ls=':', lw=1, label='1991-2020 average')

        ax.yaxis.set_major_formatter(doy_formatter())
        ax.set_ylabel(f'{label}\n(calendar day)')
        ax.set_xlabel('Year (t)')
        ax.set_title(f'{label} vs Year  (n={len(years)} years observed)')
        ax.legend(loc='best', fontsize=8)

    subtitle = f'  \u2014  {note}' if note else ''
    fig.suptitle(f'{station}: Bloom Timing vs Year{subtitle}', fontsize=12, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.965])

    fname = os.path.join(outdir, f'{idx:03d}_{safe_stub(station)}_temporal.png')
    fig.savefig(fname)
    plt.close(fig)
    return fname


def make_phase_space_plot(station, idx, years1, doy1, years2, doy2, note, outdir):
    fig, axes = plt.subplots(2, 1, figsize=(8.5, 7.5))

    panels = [
        (axes[0], years1, doy1, 'First Bloom (x1)', 'winter'),
        (axes[1], years2, doy2, 'Full Bloom (x2)', 'autumn'),
    ]
    for ax, years, doy, label, cmap_name in panels:
        if len(years) < 2:
            ax.text(0.5, 0.5, f'Insufficient data (n={len(years)}) to estimate dx/dt',
                     ha='center', va='center', transform=ax.transAxes, fontsize=10, color='gray')
            ax.set_title(f'{label}: dx/dt vs x')
            continue

        dxdt = compute_dxdt(years, doy)
        ax.plot(doy, dxdt, '-', color='gray', lw=0.6, alpha=0.5, zorder=2)
        sc = ax.scatter(doy, dxdt, c=years, cmap=cmap_name, s=30, zorder=3, edgecolor='k', linewidth=0.3)
        ax.axhline(0, color='k', lw=0.8, alpha=0.6)

        cbar = fig.colorbar(sc, ax=ax, pad=0.01)
        cbar.set_label('Year (t)')

        ax.xaxis.set_major_formatter(doy_formatter())
        ax.set_xlabel(f'x = {label} (calendar day)')
        ax.set_ylabel('dx/dt  (days/year)')
        ax.set_title(f'{label}: Phase Space (dx/dt vs x)  (n={len(years)} points)')

    subtitle = f'  \u2014  {note}' if note else ''
    fig.suptitle(f'{station}: Phase-Space Portrait{subtitle}', fontsize=12, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.965])

    fname = os.path.join(outdir, f'{idx:03d}_{safe_stub(station)}_phase_space.png')
    fig.savefig(fname)
    plt.close(fig)
    return fname


## 5. Load data and run for all stations

In [ ]:
df1 = pd.read_csv(FIRST_BLOOM_CSV)   # first-bloom dates
df2 = pd.read_csv(FULL_BLOOM_CSV)    # full-bloom dates

assert (df1['Site Name'] == df2['Site Name']).all(), \
    "Station name/order mismatch between the first-bloom and full-bloom CSVs."

year_cols = get_year_columns(df1)
temporal_dir = os.path.join(OUTPUT_DIR, 'temporal_plots')
phase_dir = os.path.join(OUTPUT_DIR, 'phase_space_plots')
os.makedirs(temporal_dir, exist_ok=True)
os.makedirs(phase_dir, exist_ok=True)

stations = df1['Site Name'].tolist()
print(f'{len(stations)} stations found. Output will be written to: {OUTPUT_DIR}/')


In [ ]:
manifest_rows = []

for idx, station in enumerate(stations, start=1):
    years1, doy1 = extract_station_series(df1, station, year_cols)
    years2, doy2 = extract_station_series(df2, station, year_cols)

    avg1 = parse_30yr_avg(df1.loc[df1['Site Name'] == station, '30 Year Average 1991-2020'].iloc[0])
    avg2 = parse_30yr_avg(df2.loc[df2['Site Name'] == station, '30 Year Average 1991-2020'].iloc[0])
    note = species_note(df1, station)

    temporal_file = make_temporal_plot(
        station, idx, years1, doy1, years2, doy2, avg1, avg2, note, temporal_dir)
    phase_file = make_phase_space_plot(
        station, idx, years1, doy1, years2, doy2, note, phase_dir)

    t1 = linear_trend(years1, doy1)
    t2 = linear_trend(years2, doy2)

    manifest_rows.append({
        'index': idx,
        'station': station,
        'currently_observed': bool(df1.loc[df1['Site Name'] == station, 'Currently Being Observed'].iloc[0]),
        'n_first_bloom_points': len(years1),
        'n_full_bloom_points': len(years2),
        'first_bloom_year_range': f'{years1.min()}-{years1.max()}' if len(years1) else '',
        'full_bloom_year_range': f'{years2.min()}-{years2.max()}' if len(years2) else '',
        'first_bloom_trend_days_per_decade': round(t1[0], 3) if t1 else np.nan,
        'full_bloom_trend_days_per_decade': round(t2[0], 3) if t2 else np.nan,
        'species_note': note or '',
        'temporal_plot': os.path.relpath(temporal_file, OUTPUT_DIR),
        'phase_space_plot': os.path.relpath(phase_file, OUTPUT_DIR),
    })

    print(f'[{idx:3d}/{len(stations)}] {station:20s} n1={len(years1):2d} n2={len(years2):2d}  -> plots saved')

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = os.path.join(OUTPUT_DIR, 'plot_manifest.csv')
manifest_df.to_csv(manifest_path, index=False)
print(f'\nAll done. Manifest written to {manifest_path}')


## 6. Quick look at the manifest

In [ ]:
manifest_df.sort_values('first_bloom_trend_days_per_decade').head(10)


## 7. Preview one station's plots inline (optional sanity check)

In [ ]:
from IPython.display import Image, display

preview_station_idx = 66  # Kyoto, by default -- change to preview a different station
row = manifest_df.loc[manifest_df['index'] == preview_station_idx].iloc[0]
display(Image(filename=os.path.join(OUTPUT_DIR, row['temporal_plot'])))
display(Image(filename=os.path.join(OUTPUT_DIR, row['phase_space_plot'])))
